# AFS-VFM Full Benchmark — Google Colab

This notebook runs the complete AFS-VFM benchmark on a **free T4 GPU**.

**Before running:** Make sure to set the runtime to GPU:
`Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Step 1: Check GPU is available ──
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Step 2: Clone the repo ──
!git clone https://github.com/AmanDevNet/AFS-VFM.git
%cd AFS-VFM

In [ ]:
# ── Step 3: Install dependencies ──
!pip install -q -r requirements.txt timm datasets huggingface_hub

In [ ]:
# ── Step 4: Login to HuggingFace (for ImageNet access) ──
# Paste your HuggingFace token below (get it from https://huggingface.co/settings/tokens)
from huggingface_hub import login
login(token="PASTE_YOUR_HF_TOKEN_HERE")  # <-- Replace this with your actual token

In [ ]:
# ── Step 5: Download datasets (COCO + ImageNet) ──
!python -u download_dataset.py

In [ ]:
# ── Step 6: Run the FULL benchmark on GPU 🚀 ──
# This will process all 1500 images × 5 degradations × 20 frames × 3 models
# Estimated time on T4 GPU: ~3-4 hours
!python -u main.py

In [ ]:
# ── Step 7: Download the results ──
from google.colab import files
files.download('results/full_benchmark.csv')
print("\nDone! Check your Downloads folder for full_benchmark.csv")

In [ ]:
# ── Step 8 (Optional): Quick stats on the results ──
import pandas as pd

df = pd.read_csv('results/full_benchmark.csv')
print(f"Total rows: {len(df)}")
print(f"\nFailure rates by model:")
print(f"  DINOv2: {df['dinov2_failure'].mean()*100:.1f}%")
print(f"  CLIP:   {df['clip_failure'].mean()*100:.1f}%")
print(f"  DETR:   {df['detr_failure'].mean()*100:.1f}%")
print(f"\nFailure rates by degradation type:")
for deg in df['degradation'].unique():
    sub = df[df['degradation'] == deg]
    print(f"  {deg:12s} — DINOv2: {sub['dinov2_failure'].mean()*100:.1f}%  CLIP: {sub['clip_failure'].mean()*100:.1f}%  DETR: {sub['detr_failure'].mean()*100:.1f}%")